# RLCD Fine-Tuning Benchmark on Kaggle (2x T4 GPU)

This notebook runs the `rlcd/main.py` pipeline from the `kaggle_benchmarking` branch across:
- **Models (2)**:
  1. **Decision 1.0**: `vllm-sr/Decision-1.0-Lex-0.6B`
  2. **Typical**: `OzLabs/typical-small`
- **Datasets (4)**:
  1. **Function Calling (`choice`)**: `Nexusflow/NexusRaven_API_evaluation`
  2. **Classification (`choice`)**: `mteb/tweet_sentiment_extraction`
  3. **Safety / Guardrail (`noul`)**: `Anthropic/hh-rlhf`
  4. **Toxicity Rating (`score`)**: `lmsys/toxic-chat`

**Kaggle Setup:** In the right sidebar under **Session options -> Accelerator**, select **GPU T4 x2** and enable **Internet**.

In [ ]:
# 1. Clone the `kaggle_benchmarking` branch of the repository and install dependencies
!rm -rf /kaggle/working/fine_tuner
!git clone -b kaggle_benchmarking https://github.com/AmeyaPadwad/fine_tuner.git /kaggle/working/fine_tuner
%cd /kaggle/working/fine_tuner/rlcd

!pip install -q -r requirements.txt scikit-learn
!nvidia-smi

Cloning into '/kaggle/working/fine_tuner'...
Successfully cloned repository and installed dependencies.
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2       |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P0             26W /  70W  |       0MiB /  15360MiB |      0%      Default |
|-----------------------------------------+------------------------+----------------------+
|   1  Tesla T4                       Off |   00000000:00:05.0 Off |

In [ ]:
# 2. (Optional) Configure Hugging Face Token from Kaggle Secrets
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF_TOKEN from Kaggle Secrets.")
except Exception:
    print("No Kaggle Secret 'HF_TOKEN' found; using public datasets.")

# Use /tmp/.cache so downloaded base weights do not consume /kaggle/working's 20GB quota
os.environ["DEFAULT_CACHE_DIR"] = "/tmp/.cache"

No Kaggle Secret 'HF_TOKEN' found; using public datasets.
Configured DEFAULT_CACHE_DIR=/tmp/.cache


In [ ]:
# 3. Models and Datasets Configuration
MODELS = [
    ("typical", "OzLabs/typical-small"),
]

DATASETS = [
    {
        "short_name": "nexusraven_choice",
        "dataset_id": "Nexusflow/NexusRaven_API_evaluation",
        "column_map": None,
    },
    {
        "short_name": "tweet_sentiment_choice",
        "dataset_id": "mteb/tweet_sentiment_extraction",
        "column_map": None,
    },
    {
        "short_name": "hh_rlhf_noul",
        "dataset_id": "Anthropic/hh-rlhf",
        "column_map": None,
    },
    {
        "short_name": "toxic_chat_score",
        "dataset_id": "lmsys/toxic-chat",
        "column_map": {
            "context": ["user_input", "model_output"],
            "target": "openai_moderation",
            "decision_type": "score",
            "scale": [1, 5],
        },
    },
]

DATASET_SUBSET = 30000
USE_MULTI_GPU_TORCHRUN = False


In [ ]:
# 4. Run main.py across both Models x all 4 Datasets and collect evaluation metrics
import json
import subprocess
import torch
import pandas as pd
from main import main as run_rlcd_main

n_gpus = torch.cuda.device_count()
print(f"Detected {n_gpus} CUDA GPU(s).")

summary_rows = []

for model_tag, model_id in MODELS:
    for ds in DATASETS:
        ds_tag = ds["short_name"]
        ds_id = ds["dataset_id"]
        col_map = ds["column_map"]
        out_dir = f"/kaggle/working/outputs/{model_tag}__{ds_tag}"

        print("\n" + "=" * 80)
        print(f"RUNNING: Model={model_id} | Dataset={ds_id} -> {out_dir}")
        print("=" * 80)

        if USE_MULTI_GPU_TORCHRUN and n_gpus > 1:
            cmd = [
                "torchrun",
                f"--nproc_per_node={n_gpus}",
                "main.py",
                "--model", model_id,
                "--dataset", ds_id,
                "--output-dir", out_dir,
            ]
            if DATASET_SUBSET is not None:
                cmd += ["--subset", str(DATASET_SUBSET)]
            if col_map is not None:
                cmd += ["--column-map", json.dumps(col_map)]
            # Stream stdout & stderr live so any child traceback is visible in the notebook
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in proc.stdout:
                print(line, end="")
            ret = proc.wait()
            if ret != 0:
                raise RuntimeError(f"torchrun failed with exit code {ret} for {model_id} on {ds_id}")
        else:
            run_rlcd_main(
                model_name=model_id,
                dataset_name=ds_id,
                dataset_subset=DATASET_SUBSET,
                column_map=col_map,
                output_dir=out_dir,
            )

        metrics_file = f"{out_dir}/test_evaluation_metrics.json"
        if os.path.exists(metrics_file):
            with open(metrics_file, "r") as f:
                m = json.load(f)
            base_m = m.get("baseline", {})
            final_m = m.get("final", {})
            summary_rows.append({
                "model": model_id,
                "dataset": ds_id,
                "task": ds_tag,
                "base_acc": base_m.get("accuracy"),
                "final_acc": final_m.get("accuracy"),
                "base_brier": base_m.get("mean_brier"),
                "final_brier": final_m.get("mean_brier"),
                "base_ece": base_m.get("ece"),
                "final_ece": final_m.get("ece"),
            })
        torch.cuda.empty_cache()

Detected 2 CUDA GPU(s).

RUNNING: Model=OzLabs/typical-small | Dataset=Nexusflow/NexusRaven_API_evaluation -> /kaggle/working/outputs/typical__nexusraven_choice
No HF_TOKEN found in .env; using cached local Hugging Face credentials if available.

=== Configuring RL Training Hyperparameters ===
Found 'OzLabs/typical-small' already cached at /tmp/.cache/typical/OzLabs/typical-small/, skipping download.
Building Typical model from '/tmp/.cache/typical/OzLabs/typical-small/best.pt'...
[transformers]  is deprecated! Use  instead!
Loading weights: 100% 310/310 [00:01<00:00, 190.18it/s]
Loaded 'OzLabs/typical-small' for fine-tuning (device='cuda').

=== Loading 'Nexusflow/NexusRaven_API_evaluation' dataset ===
Loaded Nexusflow/NexusRaven_API_evaluation...

=== Preprocessing dataset into model format ===
Preparing dataset (mode='standard', decision_type='choice', context=['prompt'], target='python_function_name')...
Map: 100% 318/318 [00:00<00:00, 6275.91 examples/s]
Dataset is now model-input

In [ ]:
# 5. Display and Save Combined Benchmark Results Table
df_results = pd.DataFrame(summary_rows)
df_results.to_csv("/kaggle/working/outputs/benchmark_summary.csv", index=False)
display(df_results)

   model                 dataset                              task                    base_acc  final_acc  base_brier  final_brier  base_ece  final_ece
0  OzLabs/typical-small  Nexusflow/NexusRaven_API_evaluation  nexusraven_choice       0.158730  0.158730   -0.907352   -0.800704    0.311367  0.154751
1  OzLabs/typical-small  mteb/tweet_sentiment_extraction     tweet_sentiment_choice  0.395436  0.395436   -0.671484   -0.660343    0.086336  0.004990
2  OzLabs/typical-small  Anthropic/hh-rlhf                    hh_rlhf_noul            0.502083  0.497917   -0.504355   -0.500017    0.046702  0.003584
3  OzLabs/typical-small  lmsys/toxic-chat                     toxic_chat_score        0.003937  0.948819   -0.930110   -0.102357    0.257833  0.051145
